# CEM4644 - MP4: Segmentation for a quantity take-off

## Homework (individual): *Seven sheets from three disciplines*

**No coding needed.** Each grey box below is one *step*: click the (play) button at its left, wait until it finishes,
look at the result, then answer the report question that follows. Run the steps **from top to bottom**.

**What you will do (about 150 minutes)**
1. Look at the seven drawings: what each one is, what it shows, and what you have to take off from it.
2. Do a take-off on each discipline in turn - floor plans, structural plans, MEP plans - with the same one cell, then
   ask SAM 3 for the same things by phrase and see which way gets you a usable number.
3. Try a drawing of your own.

**Before you start:** menu *Runtime -> Change runtime type -> T4 GPU -> Save*. The model used here (SAM 3) is large:
with a GPU each request takes well under a second; without one the steps that need the model take about a minute each.

Everything is in **feet and square feet**. There is no scale printed on a drawing that you can trust blindly: you set
the scale yourself, from a dimension the drawing prints or from something whose real size you know.

In [ ]:
#@title ▶ Step 0 · Run me first (2-3 minutes) { display-mode: "form" }
#@markdown Click the play button and wait for the 'Ready' line. This downloads the drawings with their answer keys and loads SAM 3 (about 3 GB).
#@markdown Untick *load_model* only if you have no GPU and want to skip the steps that need the live model.
load_model = True #@param {type:"boolean"}
import importlib, os, shutil, subprocess, sys
REPO, FOLDER, PKG = "CEM4644", "mp4_segmentation", "aec_seg"
FOLDERS = ["mp4_segmentation"]               # only this lab folder is downloaded, not the whole course repository

def _git(*args):
    return subprocess.run(["git", "-C", REPO, *args], capture_output=True, text=True).returncode == 0

if os.path.isdir(REPO):                      # a copy is already here: pull the newest course code over it
    if not (_git("sparse-checkout", "set", *FOLDERS)     # also trims a full copy left by an earlier run
            and _git("fetch", "-q", "--depth", "1", "origin", "master")
            and _git("reset", "-q", "--hard", "FETCH_HEAD") and _git("clean", "-qfd")):
        shutil.rmtree(REPO, ignore_errors=True)          # broken copy: start again from scratch
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "--filter=blob:none", "--sparse", "https://github.com/Haolan-Zhang/CEM4644.git", REPO], check=True)
    subprocess.run(["git", "-C", REPO, "sparse-checkout", "set", *FOLDERS], check=True)
for _m in [m for m in list(sys.modules) if m == PKG or m.startswith(PKG + ".")]:
    del sys.modules[_m]                      # Python caches imported code: drop it, or this cell keeps the old version
importlib.invalidate_caches()
sys.path.insert(0, os.path.abspath(os.path.join(REPO, FOLDER)))
from aec_seg import lab
lab.setup(dataset="homework", load_model=load_model)


## Part 1 · The drawings

Seven real drawings: two floor plans (a 1940 farmhouse and a modern VA clinic), four structural foundation plans and one reflected ceiling plan. Each one carries a different take-off task - areas of rooms, areas of footings, counts of repeated symbols - and each one has an answer key.

You already met SAM 3 in the workshop, so this notebook goes straight to the work. What stays the same on every sheet:

- **you** set the scale, from a dimension the sheet prints or from something whose size the sheet tells you. Never from a
  scale bar you have not checked.
- **you** draw the boxes. The model turns a box into an outline; it does not know what a footing or a diffuser is.
- **counts come from the model, never from a tally of your own boxes**: where a sheet asks for a count you box ONE
  example of the symbol, labelled *example: ...*, and SAM 3 finds all the others like it.
- everything you measure and everything you count is checked against an answer key, so you always see how far off you are.

In [ ]:
#@title ▶ Step 1a · Browse the seven drawings { display-mode: "form" }
#@markdown *all drawings* shows all seven; pick one to see it large.
drawing = "all drawings" #@param ["all drawings", "usda_5542: Five-room farmhouse, 32 ft x 29 ft (USDA design 710-5542)", "va_floor: VA outpatient clinic, lease module: floor plan (rooms and plumbing fixtures)", "va_ceiling: VA outpatient clinic, lease module: reflected ceiling plan (light fixture count)", "test_fp: Foundation plan: 12 ft square spread footings around an equipment pit", "test_fp_2: Foundation plan with square spread footings and an elevator shaft", "uscg_motorpool: USCG motor pool shop building: foundation plan with footing schedule", "uscg_pile: USCG bowling facility: pile footing plan (counting only)"]
lab.show_sheets(drawing)


In [ ]:
#@title ▶ Step 1b · The symbol legend { display-mode: "form" }
#@markdown The MEP sheets carry a legend of their symbols: a bold rectangle with a diagonal and a small circle is a light fixture (2 ft x 4 ft or 2 ft x 2 ft), a small circle with a cross is a recessed light. The structural sheets name their footings in a schedule printed on the sheet instead.
drawing = "all" #@param ["all", "usda_5542: Five-room farmhouse, 32 ft x 29 ft (USDA design 710-5542)", "va_floor: VA outpatient clinic, lease module: floor plan (rooms and plumbing fixtures)", "va_ceiling: VA outpatient clinic, lease module: reflected ceiling plan (light fixture count)", "test_fp: Foundation plan: 12 ft square spread footings around an equipment pit", "test_fp_2: Foundation plan with square spread footings and an elevator shaft", "uscg_motorpool: USCG motor pool shop building: foundation plan with footing schedule", "uscg_pile: USCG bowling facility: pile footing plan (counting only)"]
lab.show_legend(drawing)


## Two ways of asking, on every discipline

Each of the next three parts has two cells. The first is the **take-off**: pick the drawing, then pick the label above
the picture before each box you draw (the labels come from that sheet's answer key), then *Submit*. Three kinds of
label: **scale: ...** for a length whose size the sheet gives you, the plain word (**room**, **footing**, **pit**) for
something you want the area of, and **example: ...** for something you want counted. One box labelled *example: pile
footing* is all a count needs: SAM 3 goes and finds every other symbol on the sheet that looks like it, and that is how
a count of 41 light fixtures or 29 pile footings is made. Where the thing counted is also measured (the footings), there is
no separate example label: your **first** *footing* box is the example. The slider under the picture sets how sure the model has to be
before it keeps one of them.

The second cell is the other way of asking: **type a phrase** and SAM 3 looks for it on the whole sheet, with no box
from you at all. The regions are scored against the answer key: green for a real one found, red for one missed, blue
for a region that is not one; for rooms and footings the notebook also measures every region the phrase found. Try the trade word first (*footing*, *light fixture*), then a word for the **shape on the paper**
(*square*, *circle*, *rectangle with a diagonal line*), and move the confidence. Both cells feed the same report
question: which way gets you a number you would put in an estimate, and which way is quicker?

## Part 2 · Floor plans

On a floor plan you take off **areas of rooms**. Set the scale from a printed dimension, never from a scale bar: one of these two sheets carries a graphic scale bar that is wrong by a factor of two, and boxing it as well as the printed dimension is how you find that out. Then box every room the task asks for. The number you get is the *net* floor area: what the drawing puts on the floor (a counter, a bathtub) is cut out of the mask unless the room is a plain rectangle.

Do every drawing in the list, one at a time, in Step 2a. Phrases to try in Step 2b: Try *room*, *bedroom*, *bathroom*, *door*, *window*; then *curved line* (a door swing is an arc on the paper).

In [ ]:
#@title ▶ Step 2a · Take-off on a floor plan: your boxes { display-mode: "form" }
drawing = "usda_5542: Five-room farmhouse, 32 ft x 29 ft (USDA design 710-5542)" #@param ["usda_5542: Five-room farmhouse, 32 ft x 29 ft (USDA design 710-5542)", "va_floor: VA outpatient clinic, lease module: floor plan (rooms and plumbing fixtures)"]
lab.takeoff(drawing)


In [ ]:
#@title ▶ Step 2b · Ask by name on a floor plan: a phrase { display-mode: "form" }
drawing = "usda_5542: Five-room farmhouse, 32 ft x 29 ft (USDA design 710-5542)" #@param ["usda_5542: Five-room farmhouse, 32 ft x 29 ft (USDA design 710-5542)", "va_floor: VA outpatient clinic, lease module: floor plan (rooms and plumbing fixtures)"]
phrase = "room" #@param {type:"string"}
confidence = 0.3 #@param {type:"slider", min:0.1, max:0.9, step:0.05}
lab.ask(drawing, phrase, confidence)


> ### 📝 Report question 1
> From Step 2a: your scale reading on each sheet and how far it is from the answer key. On the VA clinic sheet you were asked to box the graphic scale bar as well as the printed dimension - what did the two give, and which one is right? (Work out what the areas would have been if you had trusted the bar.) Then the room table of one sheet: your square feet, the drawing's, the error. Which rooms are worst and why? Then from Step 2b: what did *room* find on each sheet (found / missed / extra, and the median error of the areas it measured) against the rooms from your boxes? What did *door* and *window* return, and what did *curved line*?

## Part 3 · Structural (foundation) plans

On a foundation plan you take off **areas of footings** and a **count of them**. The scale comes from a printed bay dimension or from a footing whose width its mark gives you (F4.0 = 4'-0" wide). The count is made by the model from your first *footing* box, which it uses as the example - you never count them yourself. Two things to watch: read the *same* edge of the ink at both ends (outside-to-outside or centre-to-centre, not one of each), and remember that a symbol smaller than about 60 pixels on the sheet is too small for the model - the mask becomes a rounded copy of your box, and the 'area' you get is the area you drew.

Do every drawing in the list, one at a time, in Step 3a. Phrases to try in Step 3b: Try *footing*, *foundation*, *column*; then *square* and *hatched square* for the footings, *rectangle* for the pile caps, *circle* for the grid bubbles. Move the confidence: the extras go first, then the real ones.

In [ ]:
#@title ▶ Step 3a · Take-off on a structural plan: your boxes { display-mode: "form" }
drawing = "test_fp: Foundation plan: 12 ft square spread footings around an equipment pit" #@param ["test_fp: Foundation plan: 12 ft square spread footings around an equipment pit", "test_fp_2: Foundation plan with square spread footings and an elevator shaft", "uscg_motorpool: USCG motor pool shop building: foundation plan with footing schedule", "uscg_pile: USCG bowling facility: pile footing plan (counting only)"]
lab.takeoff(drawing)


In [ ]:
#@title ▶ Step 3b · Ask by name on a structural plan: a phrase { display-mode: "form" }
drawing = "test_fp: Foundation plan: 12 ft square spread footings around an equipment pit" #@param ["test_fp: Foundation plan: 12 ft square spread footings around an equipment pit", "test_fp_2: Foundation plan with square spread footings and an elevator shaft", "uscg_motorpool: USCG motor pool shop building: foundation plan with footing schedule", "uscg_pile: USCG bowling facility: pile footing plan (counting only)"]
phrase = "footing" #@param {type:"string"}
confidence = 0.3 #@param {type:"slider", min:0.1, max:0.9, step:0.05}
lab.ask(drawing, phrase, confidence)


> ### 📝 Report question 2
> From Step 3a: for each sheet, the scale and how you set it, the areas of the footings you measured against the sizes their marks give, and the count SAM 3 made from your first *footing* box, its example (found / missed / extra). One sheet asks you only to count, because its pile caps are too small to measure - what happens to the measured area of something that small, and why? Then from Step 3b: does *footing* find anything? Which shape word finds the footings, on which sheet, with how many extras, and how do the areas it measures compare with the ones from your boxes? Did any phrase find the grid bubbles or the pile caps?

## Part 4 · MEP plans

On an MEP sheet almost nothing is measured and almost everything is **counted**. The trade words - *light fixture*, *diffuser*, *sprinkler* - return nothing at all from the model, so counting is done the other way round: box **one** example of the symbol, labelled *example: 2x4 light fixture*, and the model finds every other symbol like it. Pick an example that is clean and lying the same way as most of the others; a rotated example loses about 40 % of the count.

Do every drawing in the list, one at a time, in Step 4a. Phrases to try in Step 4b: Try *light fixture*, *light*, *diffuser*, *sprinkler*; then *rectangle with a diagonal line*, *small circle*, *circle with a cross*.

In [ ]:
#@title ▶ Step 4a · Take-off on an MEP sheet: your boxes { display-mode: "form" }
drawing = "va_ceiling: VA outpatient clinic, lease module: reflected ceiling plan (light fixture count)" #@param ["va_ceiling: VA outpatient clinic, lease module: reflected ceiling plan (light fixture count)"]
lab.takeoff(drawing)


In [ ]:
#@title ▶ Step 4b · Ask by name on an MEP sheet: a phrase { display-mode: "form" }
drawing = "va_ceiling: VA outpatient clinic, lease module: reflected ceiling plan (light fixture count)" #@param ["va_ceiling: VA outpatient clinic, lease module: reflected ceiling plan (light fixture count)"]
phrase = "light fixture" #@param {type:"string"}
confidence = 0.3 #@param {type:"slider", min:0.1, max:0.9, step:0.05}
lab.ask(drawing, phrase, confidence)


> ### 📝 Report question 3
> From Step 4a: the count of each symbol that SAM 3 made from your one example box (found / missed / extra), the confidence you used, and what the extras were. Try a second example box of the same symbol, one that is rotated or sits in a cluttered spot, and report how the count changes. Then from Step 4b: which words find any light fixture at all - the trade words or the shape words - and does the best phrase get anywhere near the count from your one example box (found / missed / extra for both)? Why do *light fixture* and *diffuser* return nothing on a drawing?

## Part 5 · Your own drawing

In [ ]:
#@title ▶ Your drawing, your words { display-mode: "form" }
#@markdown This cell prints a **link**: open it in a new tab (it works on a phone too). Upload a drawing, then ask the three ways of this lab: a box for the scale (a printed dimension, plus its length in feet), a box for a room or an object, one example box that SAM 3 finds the rest of, or a phrase. A box is two clicks on the drawing: top-left, then bottom-right. Test at least three drawings of your own, from at least two disciplines, and take screenshots. Needs the live model.
lab.upload_app()


> ### 📝 Report question 4
> Test three drawings of your own, from at least two disciplines (a floor plan, a structural plan, an MEP sheet, a section, a site plan...). For each: the phrase or box you used, what came back, and whether it is right. Which kind of drawing failed, and can you say why?

> ### 📝 Report question 5
> Across the three disciplines: which take-off task was the most reliable and which the least, and what decides that (the size of the thing on the sheet, how often it repeats, whether it is drawn as a simple outline)? Boxes or phrases: for each discipline, say which way you would use and why. Where would you use this in practice, where would it mislead you, and what would you insist on having (a known dimension, a schedule, a second pair of eyes) before putting these numbers in an estimate?

## Wrap-up

In [ ]:
#@title ▶ Numbers for your report { display-mode: "form" }
#@markdown Every take-off you submitted, printed again in one place, plus the drawings you have not done yet.
lab.report_summary()


### Where the drawings come from, and the model

- **usda_5542** - USDA design 710-5542, 'Five-room farmhouse', Miscellaneous Publication 360 'Plans of Farm Buildings for Southern States' (1940), p. 15. U.S. Department of Agriculture, Bureau of Agricultural Engineering. Public domain (work of the U.S. Government, 17 U.S.C. 105). https://archive.org/details/plansoffarmbuild360unit
- **va_floor** - Outpatient / PACT Clinic, lease module (CBOC-L.pdf), VA Technical Information Library room templates. US Department of Veterans Affairs, Office of Construction & Facilities Management (2018). Public domain (work of the US federal government). https://www.cfm.va.gov/til/rTemplate/documents/CBOC-L.pdf
- **va_ceiling** - Outpatient / PACT Clinic, lease module (CBOC-L.pdf), VA Technical Information Library room templates. US Department of Veterans Affairs, Office of Construction & Facilities Management (2018). Public domain (work of the US federal government). https://www.cfm.va.gov/til/rTemplate/documents/CBOC-L.pdf
- **test_fp** - Foundation plan, example image 1 (test_fp.png), cropped to the two interior footing rows. Course example drawing (GitHub repository Haolan-Zhang/SAM_Example_Image). Provided for this course. https://github.com/Haolan-Zhang/SAM_Example_Image
- **test_fp_2** - Foundation plan, example image 2 (test_fp_2.png). Course example drawing (GitHub repository Haolan-Zhang/SAM_Example_Image). Provided for this course. https://github.com/Haolan-Zhang/SAM_Example_Image
- **uscg_motorpool** - Motor Pool Facility, Support Center New York, Shop Building (Building 928): structural foundation plan, details and notes, sheet S-1, 20 June 1984. U.S. Coast Guard, 3rd District Governors Island NY, Civil Engineering; drawn by Gonchor & Sput, Architects & Planners. Public domain (work of the US federal government). https://commons.wikimedia.org/wiki/File:Building_928_Structural_Foundation_Plan,_Details_and_Notes_Shop_Building,_June_20,_1984_-_DPLA_-_2938a6cfb0663504470f4b44fe7ae27e.tiff
- **uscg_pile** - Building 785, Sixteen Lane Bowling Facility, Governors Island: foundation plan, sheet 7 of 11, 19 January 1983. U.S. Coast Guard Support Center New York, Public Works Engineering (traced from International Design Consultants Inc.). Public domain (work of the US federal government). https://commons.wikimedia.org/wiki/File:Building_785_Sixteen_Lane_Bowling_Facility_Foundation_Plan,_January_19,_1983_-_DPLA_-_ec587aa2a393d5917cb73be19a2da195.tiff
- Model: SAM 3 by Meta AI (SAM License), loaded from a public mirror of the official checkpoint; a copy of the licence is in `docs/SAM_LICENSE.txt`.
- Lab code: https://github.com/Haolan-Zhang/CEM4644 (folder `mp4_segmentation`).